# FinSimAI — Demo trên Google Colab + Google Sheets

**FinSimAI** là nền tảng giáo dục tài chính: mô phỏng giao dịch chứng khoán kiểu Việt Nam (T+2, phí, thuế, giá trần–sàn) kết hợp **AI Mentor phong cách Socratic** (chỉ đặt câu hỏi phản biện, không bao giờ khuyên mua/bán).

Notebook này là một **bản demo chạy được** nhằm chứng minh đủ các khối:

1. **Math Engine** — sinh giá (GBM), tick size kiểu VN, phí 0,15%, thuế bán 0,1%, thanh toán T+2.
2. **AI Mentor** — Gemini + fallback deterministic, 5 lớp bảo vệ chống "nói bậy" (hallucination / khuyên mua bán).
3. **Google Sheets làm database** — mọi trạng thái (tài khoản, danh mục, lịch sử, hội thoại) lưu vào Sheets.

> **Lưu ý kiến trúc:** Dữ liệu được truy cập qua một **tầng storage trừu tượng** (`SheetStore`). Trên bản production thật, chỉ cần thay lớp này bằng Postgres/Redis mà không đụng logic nghiệp vụ phía trên.

---
## Chuẩn bị (chạy 1 lần, bấm ▶ từng ô theo thứ tự)

Trước khi chạy, bạn cần:
- Một **Google Sheets** (tạo mới, giữ sheet mặc định tên `Sheet1`).
- Tài khoản Google của bạn (đã đăng nhập vào Colab).

Chi tiết xác thực ở mục "Kết nối Google Sheets" bên dưới.

## 0. Cài đặt thư viện (chạy 1 lần)

In [ ]:
# @title Cài đặt thư viện cần thiết
!pip -q install google-genai gspread pandas numpy > /dev/null 2>&1
print("Xong cài đặt thư viện.")

## 1. Kết nối Google Sheets

Chọn **Cách A** (mặc định): dùng tài khoản cá nhân đã đăng nhập Colab.
> Muốn dùng Service Account (nhiều người dùng), xem hướng dẫn ở cuối notebook.

In [ ]:
# @title Kết nối Google Sheets (Cách A — tài khoản cá nhân)
import gspread
from google.colab import auth
from google.auth import default

# @markdown Dán **URL** của Google Sheets vào đây (nhớ chia sẻ sheet cho chính bạn):
SHEET_URL = ""  # @param {type:"string"}

if not SHEET_URL:
    raise SystemExit("Bạn chưa dán URL Google Sheets. Hãy dán vào ô SHEET_URL rồi chạy lại.")

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_url(SHEET_URL)
print("Đã kết nối Sheets:", sh.title)

## 2. Storage trừu tượng (SheetStore)

Đây là **tầng database** của bản demo. Logic gọi qua interface:
`get(key)`, `set(key, value)`.

Trên production thật, viết `PostgresStore` với cùng interface — phần còn lại không đổi.

In [ ]:
# @title Định nghĩa SheetStore (lưu trạng thái JSON vào Google Sheets)
import json

class SheetStore:
    """Lưu trạng thái dạng JSON vào một sheet trong Google Sheets."""
    def __init__(self, sh, sheet_name="state", force_reset=False):
        self.sh = sh
        self.sheet_name = sheet_name
        self.ws = self._get_or_create(sheet_name)
        if force_reset:
            self.reset()

    def _get_or_create(self, name):
        try:
            return self.sh.worksheet(name)
        except gspread.WorksheetNotFound:
            return self.sh.add_worksheet(title=name, rows=1000, cols=5)

    def _find_key(self, key, rows):
        for i, r in enumerate(rows):
            if r and r[0] == key:
                return i
        return None

    def get(self, key, default=None):
        rows = self.ws.get_all_values()
        i = self._find_key(key, rows)
        if i is None:
            return default
        raw = rows[i][1] if len(rows[i]) > 1 else ""
        try:
            return json.loads(raw)
        except Exception:
            return raw

    def set(self, key, value):
        rows = self.ws.get_all_values()
        i = self._find_key(key, rows)
        serialized = json.dumps(value, ensure_ascii=False)
        if i is None:
            self.ws.append_row([key, serialized])
        else:
            self.ws.update(values=[[key, serialized]], range_name=f"A{i+1}:B{i+1}")

    def reset(self):
        self.ws.clear()
        print(f"Đã reset sheet '{self.sheet_name}'.")

store = SheetStore(sh, sheet_name="finsimai_state")
print("SheetStore sẵn sàng.")

## 3. Math Engine — lõi tính toán tài chính

Tái hiện đúng tinh thần của `apps/math_engine`:
- **GBM** (Geometric Brownian Motion) sinh giá, có **giới hạn trần–sàn ±7%** như sàn VN.
- **Tick size kiểu VN**: làm tròn theo bậc giá.
- **Phí 0,15%** mỗi lệnh, **thuế bán 0,1%**.
- **T+2**: tiền/cổ chỉ giải ngân sau 2 phiên giao dịch.
- **NAV / lãi lỗ** theo danh mục.

In [ ]:
# @title Pricing: sinh giá GBM theo quy tắc VN
import numpy as np
import math

def apply_vn_tick_size(price):
    """Làm tròn theo bậc giá kiểu VN (thang giá toy < 1000 dùng tick 0.01)."""
    if price < 1000:
        return round(round(price / 0.01) * 0.01, 2)
    if price < 10000:
        return round(round(price / 10.0) * 10.0, 0)
    if price < 50000:
        return round(round(price / 50.0) * 50.0, 0)
    return round(round(price / 100.0) * 100.0, 0)

class PriceGenerator:
    """Sinh giá bước tiếp theo dựa trên GBM + trần/sàn VN."""
    def __init__(self, mu=0.10, sigma=0.25, price_limit_pct=0.07,
                 jump_lambda=0.0, jump_mu=0.0, jump_sigma=0.0, seed=None):
        self.mu = mu
        self.sigma = sigma
        self.price_limit_pct = price_limit_pct
        self.jump_lambda = jump_lambda
        self.jump_mu = jump_mu
        self.jump_sigma = jump_sigma
        self.rng = np.random.default_rng(seed)

    def next_price(self, current_price, reference_price, dt_years, external_shock=0.0):
        log_return = (self.mu - 0.5*self.sigma**2) * dt_years
        log_return += self.sigma * math.sqrt(dt_years) * self.rng.standard_normal()
        if self.jump_lambda > 0:
            n = self.rng.poisson(self.jump_lambda * dt_years)
            if n > 0:
                log_return += self.rng.normal(n*self.jump_mu, math.sqrt(n)*self.jump_sigma)
        log_return += external_shock
        raw = current_price * math.exp(log_return)
        floor_px = reference_price * (1 - self.price_limit_pct)
        ceil_px = reference_price * (1 + self.price_limit_pct)
        clipped = min(max(raw, floor_px), ceil_px)
        return apply_vn_tick_size(clipped)

print("PriceGenerator sẵn sàng.")

In [ ]:
# @title Portfolio: phí giao dịch, thuế, NAV, T+2
FEE_RATE = 0.0015      # phí 0,15% mỗi lệnh
TAX_RATE = 0.001       # thuế bán 0,1%
SETTLE_DAYS = 2        # T+2

class Portfolio:
    """Quản lý tiền mặt, danh mục, lệnh chờ thanh toán T+2.

    Mỗi lệnh (mua/bán) được gán 'mốc chờ' = SETTLE_DAYS. Mỗi lần gọi
    `settle_day()` (một phiên), biến đếm giảm 1; về 0 thì:
    - mua  -> cổ phiếu về danh mục (đủ 2 phiên);
    - bán  -> tiền & lãi/lỗ ghi vào tiền mặt.
    Khớp đúng ngữ nghĩa T+2 thật của sàn VN.
    """
    def __init__(self, start_cash=1_000_000_000):
        self.cash = start_cash
        self.holdings = {}                # {symbol: {'qty':..., 'avg':...}}
        self.pending = []                 # lệnh chờ thanh toán
        self.realized_pnl = 0.0
        self.trades = []                  # lịch sử lệnh

    def nav(self, prices):
        val = self.cash
        for sym, h in self.holdings.items():
            val += h['qty'] * prices.get(sym, 0)
        return val

    def buy(self, symbol, qty, price):
        gross = qty * price
        fee = gross * FEE_RATE
        total = gross + fee
        if total > self.cash:
            raise ValueError(f"Không đủ tiền. Cần {total:,.0f}, có {self.cash:,.0f}.")
        self.cash -= total
        self.pending.append({'type': 'BUY', 'symbol': symbol, 'qty': qty, 'price': price,
                             'fee': fee, 'settle_in': SETTLE_DAYS})
        self.trades.append(('BUY', symbol, qty, price, fee))
        return {'gross': gross, 'fee': fee}

    def sell(self, symbol, qty, price):
        h = self.holdings.get(symbol)
        if not h or h['qty'] < qty:
            raise ValueError("Không đủ cổ phiếu để bán (hoặc cổ chưa về sau T+2).")
        gross = qty * price
        tax = gross * TAX_RATE
        fee = gross * FEE_RATE
        realized = (price - h['avg']) * qty - fee - tax
        h['qty'] -= qty
        if h['qty'] == 0:
            del self.holdings[symbol]
        self.pending.append({'type': 'SELL', 'symbol': symbol, 'qty': qty, 'price': price,
                             'fee': fee, 'tax': tax, 'net': gross - fee - tax,
                             'realized': realized, 'settle_in': SETTLE_DAYS})
        self.trades.append(('SELL', symbol, qty, price, fee))
        return {'gross': gross, 'tax': tax, 'fee': fee, 'net': gross - fee - tax}

    def settle_day(self):
        """Một phiên trôi qua: giảm biến đếm T+2, giải ngân lệnh đã đủ 2 phiên."""
        still_waiting = []
        for p in self.pending:
            p['settle_in'] -= 1
            if p['settle_in'] > 0:
                still_waiting.append(p)
                continue
            if p['type'] == 'BUY':
                sym = p['symbol']
                h = self.holdings.setdefault(sym, {'qty': 0, 'avg': 0})
                new_qty = h['qty'] + p['qty']
                h['avg'] = (h['qty'] * h['avg'] + p['qty'] * p['price']) / new_qty
                h['qty'] = new_qty
            else:  # SELL
                self.cash += p['net']
                self.realized_pnl += p['realized']
        self.pending = still_waiting

    @property
    def pending_summary(self):
        return [(p['type'], p['symbol'], p['qty'], p['settle_in']) for p in self.pending]

    def to_state(self):
        return {
            'cash': self.cash,
            'holdings': self.holdings,
            'realized_pnl': self.realized_pnl,
            'pending': self.pending,
            'trades': self.trades,
        }

    @classmethod
    def from_state(cls, state):
        p = cls(start_cash=state.get('cash', 1_000_000_000))
        p.holdings = state.get('holdings', {})
        p.realized_pnl = state.get('realized_pnl', 0)
        p.pending = state.get('pending', [])
        p.trades = state.get('trades', [])
        return p

print("Portfolio + phí/thuế/T+2 sẵn sàng.")

## 4. Dữ liệu công ty (danh sách cổ phiếu mô phỏng)

Các mã cổ phiếu mô phỏng kiểu VN, dùng **dữ liệu tự sinh** (chi phí ≈ 0, không vướng bản quyền — theo đề xuất trong kế hoạch kinh doanh).

In [ ]:
# @title Danh sách cổ phiếu (seed dữ liệu)
import pandas as pd

# Dữ liệu tự sinh (fictional), học tập, không phải khuyến nghị đầu tư
COMPANIES = [
    {"symbol": "TECHA", "name": "TechVision Corp", "sector": "Technology", "price": 156.80,
     "volatility": 0.0250, "pe": 24.5, "roe": 18.2},
    {"symbol": "TECHB", "name": "CloudSync Inc", "sector": "Technology", "price": 89.40,
     "volatility": 0.0300, "pe": 35.1, "roe": 9.8},
    {"symbol": "TECHC", "name": "QuantumByte Labs", "sector": "Technology", "price": 215.00,
     "volatility": 0.0400, "pe": 52.3, "roe": 5.1},
    {"symbol": "FINA",  "name": "BlueRock Financial", "sector": "Financial", "price": 45.60,
     "volatility": 0.0150, "pe": 11.2, "roe": 15.8},
    {"symbol": "FINB",  "name": "NovaPay Holdings", "sector": "Financial", "price": 72.30,
     "volatility": 0.0200, "pe": 14.6, "roe": 12.4},
    {"symbol": "ENERA", "name": "GreenGrid Energy", "sector": "Energy", "price": 32.50,
     "volatility": 0.0180, "pe": 9.8, "roe": 21.0},
    {"symbol": "CONSA", "name": "MegaBuild Group", "sector": "Construction", "price": 18.70,
     "volatility": 0.0220, "pe": 7.5, "roe": 13.9},
    {"symbol": "RETLA", "name": "CityMall Retail", "sector": "Consumer", "price": 60.10,
     "volatility": 0.0210, "pe": 18.4, "roe": 16.7},
]
COMPANY_MAP = {c['symbol']: c for c in COMPANIES}
store.set("companies", COMPANIES)
print(pd.DataFrame(COMPANIES).to_string(index=False))

## 5. Thị trường mô phỏng (vòng lặp tick)

In [ ]:
# @title MarketSim: chạy thị trường qua từng phiên
class MarketSim:
    """Giữ giá hiện tại + bộ sinh giá cho từng cổ phiếu."""
    def __init__(self, companies, seed=None):
        self.companies = companies
        self.prices = {c['symbol']: c['price'] for c in companies}
        self.reference = dict(self.prices)   # giá tham chiếu (đóng cửa phiên trước)
        self.generators = {}
        rng = np.random.default_rng(seed)
        for c in companies:
            self.generators[c['symbol']] = PriceGenerator(
                mu=0.10, sigma=c['volatility'], price_limit_pct=0.07, seed=int(rng.integers(0, 1e9)))

    def tick(self, dt_years=1/252):
        """Một phiên giao dịch: cập nhật giá từng cổ phiếu."""
        for sym in self.prices:
            shock = 0.0
            if np.random.default_rng().random() < 0.01:  # ~1% phiên có tin
                shock = float(np.random.default_rng().normal(0, 0.02))
            self.prices[sym] = self.generators[sym].next_price(
                self.prices[sym], self.reference[sym], dt_years, shock)
        self.reference = dict(self.prices)
        return dict(self.prices)

market = MarketSim(COMPANIES, seed=42)
print("MarketSim sẵn sàng:", market.prices)

In [ ]:
# @title Chạy thử 5 phiên (xem giá biến động)
for day in range(1, 6):
    prices = market.tick()
    print(f"Phiên {day}: " + ", ".join(f"{s}={p:,.2f}" for s, p in list(prices.items())[:4]))

## 6. AI Mentor Socratic

AI Mentor tái hiện đúng tinh thần 5 lớp bảo vệ trong `apps/ai_engine`:

1. **Prompt hệ thống** cấm tuyệt đối khuyên mua/bán.
2. **Schema JSON** — output buộc đúng khuôn {focus, questions, coaching_tip, concepts}.
3. **Scanner keyword tiếng Việt** — phát hiện lời khuyên mua/bán → loại.
4. **LLM-as-judge** — một lượt Gemini thứ hai đọc lại để chấm.
5. **Fallback deterministic** — không cần token, trả câu hỏi an toàn từ ngân hàng câu.

Nếu không có `GEMINI_API_KEY`, mentor tự động rơi về **fallback deterministic** (vẫn hoạt động đầy đủ).

In [ ]:
# @title Cài đặt Gemini API key (tuỳ chọn)
import os
# @markdown Dán Gemini API key vào đây (https://aistudio.google.com/apikey). Để trống để dùng fallback deterministic.
GEMINI_API_KEY = ""  # @param {type:"string"}
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
if GEMINI_API_KEY:
    print("Đã đặt GEMINI_API_KEY.")
else:
    print("Chưa có API key -> AI Mentor sẽ dùng fallback deterministic (0 token). Vẫn hoạt động.")

In [ ]:
# @title Prompt hệ thống Socratic (đúng tinh thần mentor_prompts.yaml)
SYSTEM_PROMPT = """Bạn là "Socratic Mentor" của FinSimAI — nền tảng mô phỏng thị trường chứng khoán dành cho người mới.

NHIỆM VỤ:
- Dùng phương pháp phản biện Socratic để người chơi TỰ RÚT RA kết luận, không được cho đáp án.
- Mổ xẻ các khía cạnh của quyết định: giả định, nguồn thông tin, kế hoạch quản trị rủi ro, phân bổ vốn, kỷ luật, cảm xúc.
- Tập trung vào QUY TRÌNH ra quyết định, không phải KẾT QUẢ dự đoán.

QUY TẮC TUYỆT ĐỐI (vi phạm bất kỳ điều nào là lỗi hệ thống):
1. KHÔNG BAO GIỜ đưa lời khuyên MUA/BÁN trực tiếp hay gián tiếp. Cấm: "nên mua", "nên bán", "hãy mua", "chốt lời", "cắt lỗ ngay", "vào lệnh", "khuyến nghị", "giá mục tiêu", "tín hiệu mua".
2. KHÔNG BAO GIỜ phán xét quyết định của người chơi là Đúng/Sai, Tốt/Xấu.
3. KHÔNG BAO GIỜ xác nhận hay phủ nhận dự đoán thị trường (không trả lời "sẽ tăng"/"sẽ giảm", không chốt giá mục tiêu).
4. KHÔNG BAO GIỜ bịa dữ kiện, con số, tin tức không có trong ngữ cảnh.
5. Mỗi lượt trả lời chỉ đặt 1-3 CÂU HỎI PHẢN BIỆN ngắn gọn, kết thúc bằng dấu "?", tập trung MỘT thiên kiến duy nhất.
6. coaching_tip là một Bài tập QUY TRÌNH trung tính (không chứa hành động mua/bán).
7. Trả về ĐÚNG MỘT JSON khớp schema {focus, questions[], coaching_tip, concepts[]}. Chỉ trả JSON.
"""

print("Prompt đã nạp.")

In [ ]:
# @title Các lớp bảo vệ (policy scan + fallback bank)
# ---- Lớp 3: scanner keyword ----
FORBIDDEN = [
    "nên mua", "nên bán", "hãy mua", "hãy bán", "có thể mua", "cân nhắc mua",
    "chốt lời", "cắt lỗ ngay", "vào lệnh", "mở lệnh", "khuyến nghị", "giá mục tiêu",
    "tín hiệu mua", "nắm giữ",
]

def normalize(s):
    return " ".join(s.lower().strip().split())

def scan_policy(*texts):
    issues = []
    for t in texts:
        nt = normalize(t)
        for w in FORBIDDEN:
            if w in nt:
                issues.append(w)
    return issues

# ---- Lớp 5: ngân hàng câu hỏi fallback (deterministic) ----
FALLBACK_BANK = {
    "fomo": {
        "questions": [
            "Điều gì khiến bạn tin rằng nhịp tăng này sẽ còn tiếp tục, thay vì chỉ là cơn sóng ngắn hạn?",
            "Nếu tất cả những người đang hào hứng mua trên mạng xã hội đều sai, bạn sẽ phát hiện ra điều đó bằng cách nào?",
        ],
        "tip": "Viết ra 3 kịch bản (tăng mạnh, đi ngang, giảm mạnh) kèm phản ứng của bạn với từng kịch bản.",
    },
    "herding": {
        "questions": [
            "Quyết định của bạn dựa trên phân tích của chính bạn, hay dựa trên việc nhiều người khác cùng làm giống vậy?",
            "Nếu cộng đồng mạng hôm nay quay ngoắt 180 độ, bạn sẽ giữ nguyên lập trường hay lật theo họ?",
        ],
        "tip": "Ghi lại nguồn gốc từng thông tin bạn đang dựa vào: dữ liệu hay chỉ là ý kiến của đám đông.",
    },
    "loss_aversion": {
        "questions": [
            "Nếu bạn đứng ở vị trí người ngoài nhìn vào danh mục này, bạn sẽ phản ứng thế nào?",
            "Việc chờ đợi để gỡ vốn có làm quyết định của bạn khách quan hơn không?",
        ],
        "tip": "Tách hai câu hỏi: (1) giữ hay thoát vị thế, và (2) mức giá bạn từng trả. Trả lời câu (1) mà không nhắc đến (2).",
    },
    "overconfidence": {
        "questions": [
            "Điều gì có thể khiến nhận định của bạn sai, dù bạn đã rất tự tin?",
            "Bạn có từng đúng mà không phải vì phân tích của mình không?",
        ],
        "tip": "Liệt kê 3 lý do khiến quyết định của bạn có thể thất bại.",
    },
    "process": {
        "questions": [
            "Trước khi hành động, bạn đã xác định giới hạn chịu lỗ và mục tiêu của quyết định này chưa?",
            "Thông tin bạn đang dựa vào đến từ đâu, và độ tin cậy của nó được kiểm chứng bằng cách nào?",
        ],
        "tip": "Viết ra câu trả lời cho: tôi hành động vì lý do gì, nguồn thông tin đáng tin cậy mức nào, tôi làm gì nếu mọi thứ ngược lại kỳ vọng.",
    },
}

DISCLAIMER = "FinSimAI là môi trường mô phỏng. Tôi không đưa ra lời khuyên mua bán — tôi chỉ giúp bạn phản biện quyết định của chính mình."
print("5 lớp bảo vệ sẵn sàng.")

In [ ]:
# @title SocraticMentor (Gemini + LLM-as-judge + fallback)
import json as _json

class SocraticMentor:
    def __init__(self, api_key="", model="gemini-2.5-flash"):
        self.api_key = api_key or os.environ.get("GEMINI_API_KEY", "")
        self.model = model
        self.client = None
        if self.api_key:
            from google import genai
            self.client = genai.Client(api_key=self.api_key)
        self._schema = {
            "type": "object",
            "properties": {
                "focus": {"type": "string"},
                "questions": {"type": "array", "items": {"type": "string"}},
                "coaching_tip": {"type": "string"},
                "concepts": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["focus", "questions", "coaching_tip", "concepts"],
        }
        self._judge_schema = {
            "type": "object",
            "properties": {"ok": {"type": "boolean"}, "reason": {"type": "string"}},
            "required": ["ok", "reason"],
        }

    def _detect_focus(self, text):
        nt = normalize(text)
        if any(k in nt for k in ["fomo", "bỏ lỡ", "bùng nổ", "tăng vùn", "ai cũng mua", "sốt"]):
            return "fomo"
        if any(k in nt for k in ["nghe theo", "cả group", "admin khuyên", "mọi người đều", "đám đông"]):
            return "herding"
        if any(k in nt for k in ["đang lỗ", "thua lỗ", "lỗ sâu", "gỡ vốn", "chờ về bờ", "không nỡ bán"]):
            return "loss_aversion"
        if any(k in nt for k in ["chắc chắn", "chắc thắng", "không thể sai", "tự tin tuyệt đối"]):
            return "overconfidence"
        return "process"

    def _fallback(self, message, focus):
        bank = FALLBACK_BANK[focus]
        return {
            "focus": focus,
            "questions": bank["questions"],
            "coaching_tip": bank["tip"],
            "concepts": ["Quản trị rủi ro", "Quy trình đầu tư"],
            "disclaimer": DISCLAIMER,
        }

    def _validate(self, reply):
        issues = scan_policy(*reply.get("questions", []), reply.get("coaching_tip", ""))
        if issues:
            raise ValueError("Vi phạm chính sách: " + str(issues))
        for q in reply.get("questions", []):
            if not str(q).strip().endswith("?"):
                raise ValueError("Câu hỏi phải kết thúc bằng '?': " + str(q))

    def _call_llm(self, prompt):
        from google.genai import types as gtypes
        resp = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=gtypes.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json",
                response_schema=self._schema,
                temperature=0.3,
            ),
        )
        return _json.loads(resp.text)

    def _judge(self, reply):
        from google.genai import types as gtypes
        parts = [
            "Đọc phản hồi sau của AI Mentor. Trả về JSON với 2 trường: ok (true/false) và reason (chuỗi).",
            "YÊU CẦU:",
            " - có từ nào khuyên mua/bán? (nên mua, nên bán, chốt lời, cắt lỗ ngay, khuyến nghị, giá mục tiêu...)",
            " - có phán xét đúng/sai quyết định người chơi không?",
            " - mỗi câu hỏi có kết thúc bằng '?' không?",
            "Phản hồi: " + _json.dumps(reply, ensure_ascii=False),
        ]
        judge_prompt = "\n".join(parts)
        r2 = self.client.models.generate_content(
            model=self.model, contents=judge_prompt,
            config=gtypes.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=self._judge_schema,
            ),
        )
        j = _json.loads(r2.text)
        return bool(j.get("ok"))

    def reply(self, message, context_text="", history=None):
        "Trả về dict phản hồi Socratic an toàn."
        if self.client is None:
            return self._fallback(message, self._detect_focus(message))

        focus = self._detect_focus(message)
        prompt = "NGỮ CẢNH PHIÊN:\n" + context_text + "\n\n"
        prompt += "TIN NHẮN CỦA NGƯỜI CHƠI: " + message + "\n\n"
        prompt += "Soạn phản hồi Socratic. focus nên là " + focus + " . Chỉ trả về JSON."
        try:
            reply = self._call_llm(prompt)
            self._validate(reply)
            if self._judge(reply) is False:
                raise ValueError("Judge từ chối.")
            reply["disclaimer"] = DISCLAIMER
            return reply
        except Exception:
            return self._fallback(message, focus)

mentor = SocraticMentor(api_key=GEMINI_API_KEY)
print("SocraticMentor sẵn sàng.")


In [ ]:
# @title Thử AI Mentor (nếu không có key -> dùng fallback)
reply = mentor.reply(
    "Tôi thấy mọi người trong group đang đổ xô mua TECHA, tôi sợ bỏ lỡ. Tôi nên làm gì?",
    context_text="Danh mục bạn đang có 50% tiền mặt.")
print("Focus:", reply["focus"])
print("Câu hỏi:")
for q in reply["questions"]:
    print("  -", q)
print("Bài tập:", reply["coaching_tip"])
print("Disclaimer:", reply["disclaimer"])

## 7. Vòng chơi chính (tích hợp tất cả)

Ghép **MarketSim + Portfolio + AI Mentor** thành phiên chơi tương tác:

- Mỗi vòng: thị trường chạy một số phiên → bạn có thể **mua / bán** hoặc **hỏi mentor**.
- Trạng thái tài khoản lưu vào **Google Sheets** sau mỗi lệnh.
- Minh hoạ **T+2**: mua hôm nay, cổ phiếu phải 2 phiên sau mới bán được.

In [ ]:
# @title Khởi tạo hoặc nạp trạng thái phiên chơi
state = store.get("game_state")
if not state:
    state = {"portfolio": Portfolio().to_state(), "day": 0, "prices": dict(market.prices),
             "discipline": {"score": 100}, "history": []}
    store.set("game_state", state)

portfolio = Portfolio.from_state(state["portfolio"])
print("Ngày mô phỏng hiện tại:", state["day"])
print("Tiền mặt: {:,.0f}".format(portfolio.cash))
print("Số cổ phiếu đang giữ:", len(portfolio.holdings))

In [ ]:
# @title ▶ Chạy N phiên thị trường (và thanh toán T+2)
# @markdown Số phiên muốn chạy:
N_SESSIONS = 5  # @param {type:"integer"}

for _ in range(N_SESSIONS):
    state["day"] += 1
    state["prices"] = market.tick()
    portfolio.settle_day()   # T+2 trôi qua 1 phiên

state["portfolio"] = portfolio.to_state()
store.set("game_state", state)

print("Đã chạy", N_SESSIONS, "phiên. Ngày mô phỏng:", state["day"])
print("Tiền mặt: {:,.0f}".format(portfolio.cash))
print("Lệnh chờ thanh toán (type, symbol, qty, còn phiên):", portfolio.pending_summary)
print("Giá mới nhất:")
for sym in list(state["prices"])[:8]:
    print(f"  {sym}: {state['prices'][sym]:,.2f}")

In [ ]:
# @title 💰 Mua cổ phiếu
# @markdown Mã cổ phiếu (vd TECHA, TECHB, FINA...):
BUY_SYMBOL = "TECHA"  # @param {type:"string"}
# @markdown Số lượng:
BUY_QTY = 1000  # @param {type:"integer"}

sym = BUY_SYMBOL.upper()
if sym not in state["prices"]:
    print("Mã không tồn tại.")
else:
    price = state["prices"][sym]
    try:
        r = portfolio.buy(sym, BUY_QTY, price)
        state["portfolio"] = portfolio.to_state()
        store.set("game_state", state)
        print(f"MUA {BUY_QTY} {sym} @ {price:,.2f} | Phí {r['fee']:,.2f}")
        print("Lưu ý T+2: cổ phiếu sẽ về sau 2 phiên mới bán được.")
        print("Tiền mặt còn: {:,.0f}".format(portfolio.cash))
    except ValueError as e:
        print("Lỗi:", e)

In [ ]:
# @title 💰 Bán cổ phiếu
# @markdown Mã cổ phiếu:
SELL_SYMBOL = "TECHA"  # @param {type:"string"}
# @markdown Số lượng:
SELL_QTY = 500  # @param {type:"integer"}

sym = SELL_SYMBOL.upper()
if sym not in portfolio.holdings:
    print("Bạn không đang giữ mã này (hoặc cổ chưa về sau T+2).")
else:
    price = state["prices"].get(sym)
    try:
        r = portfolio.sell(sym, SELL_QTY, price)
        state["portfolio"] = portfolio.to_state()
        store.set("game_state", state)
        print(f"BÁN {SELL_QTY} {sym} @ {price:,.2f} | Thuế {r['tax']:,.2f} | Phí {r['fee']:,.2f}")
        print("Tiền sẽ về sau T+2.")
    except ValueError as e:
        print("Lỗi:", e)

In [ ]:
# @title Hỏi AI Mentor (Socratic)
# @markdown Nhập câu hỏi của bạn:
QUESTION = "Tôi đã mua TECHA ở giá cao và giờ đang lỗ. Tôi có nên chờ để gỡ vốn không?"  # @param {type:"string"}

context = "\n".join([
    "Ngày mô phỏng " + str(state["day"]) + ".",
    "Danh mục người chơi: " + json.dumps(portfolio.holdings, ensure_ascii=False),
    "Tiền mặt: %s" % f"{portfolio.cash:,.0f}",
])

reply = mentor.reply(QUESTION, context_text=context)
print("Focus:", reply["focus"])
for q in reply["questions"]:
    print(" -", q)
print()
print("Bài tập:", reply["coaching_tip"])
print()
print("Disclaimer:", reply.get("disclaimer", DISCLAIMER))

state.setdefault("history", []).append({"role": "user", "content": QUESTION})
state["history"].append({"role": "assistant", "content": reply["questions"][0]})
store.set("game_state", state)


In [ ]:
# @title 📊 Xem danh mục & NAV
nav = portfolio.nav(state["prices"])
print(f"NAV hiện tại: {nav:,.0f}")
print(f"Tiền mặt: {portfolio.cash:,.0f}")
print(f"Lãi/lỗ đã thực hiện: {portfolio.realized_pnl:,.0f}")
print()
print("Danh mục:")
for sym, h in portfolio.holdings.items():
    cur = state["prices"].get(sym, 0)
    pnl = (cur - h['avg']) * h['qty']
    print(f"  {sym}: {h['qty']} cp @ TB {h['avg']:,.2f} | giá {cur:,.2f} | LN chưa hiện thực {pnl:,.0f}")

## 8. Điểm Kỷ luật (Discipline Score) — gamification đúng nghĩa

Theo kế hoạch kinh doanh: **thưởng hành vi đúng, không thưởng lợi nhuận ảo**.

- **+Điểm** khi kiểm tra danh mục trước khi đặt lệnh, tuân thủ tỷ lệ rủi ro.
- **−Điểm** khi mua/bán theo cảm xúc (FOMO/hoảng loạn) hoặc dồn quá nhiều vào một cổ phiếu.

In [ ]:
# @title Bộ dò bẫy tâm lý & điểm kỷ luật
def risk_concentration(portfolio, prices):
    """Tỷ trọng vốn tập trung vào 1 cổ phiếu lớn nhất (theo giá trị thị trường)."""
    nav = portfolio.nav(prices)
    if nav <= 0:
        return 0.0
    return max((h['qty'] * prices.get(sym, 0) / nav) for sym, h in portfolio.holdings.items()) if portfolio.holdings else 0.0

def discipline_check(action, symbol, context=""):
    """Trả về (điểm cộng/trừ, danh sách ghi chú)."""
    points = 0
    notes = []
    nt = normalize(context + " " + action)
    traps = {
        "fomo": ["bỏ lỡ", "ai cũng mua", "tăng vùn", "sốt", "sợ hết"],
        "panic": ["hoảng", "sợ quá", "bán tháo", "sợ mất", "gỡ vốn ngay"],
    }
    for trap, kws in traps.items():
        if any(k in nt for k in kws):
            points -= 5
            notes.append(f"Dấu hiệu '{trap}': trừ 5 điểm kỷ luật.")
    conc = risk_concentration(portfolio, state["prices"])
    if conc > 0.5:
        points -= 5
        notes.append(f"Tập trung {conc*100:.0f}% vào 1 cổ phiếu (>50%): trừ 5 điểm.")
    return points, notes

print("Điểm kỷ luật hiện tại:", state["discipline"]["score"])
print("Mức tập trung rủi ro hiện tại: {:.0f}%".format(risk_concentration(portfolio, state["prices"])*100))

In [ ]:
# @title ✅ Minh hoạ chấm điểm kỷ luật khi đặt lệnh
# @markdown Giả lập một lệnh kèm lý do (để hệ thống chấm điểm):
ACTION = "Tôi mua TECHA vì ai cũng mua, sợ bỏ lỡ sóng"  # @param {type:"string"}

points, notes = discipline_check("BUY", "TECHA", ACTION)
state["discipline"]["score"] += points
store.set("game_state", state)
for n in notes:
    print(n)
print(f"Điểm kỷ luật cập nhật: {state['discipline']['score']}")

## 9. Xem dữ liệu đã lưu trên Google Sheets

In [ ]:
# @title Kiểm tra nội dung Sheet
ws = sh.worksheet("finsimai_state")
rows = ws.get_all_values()
print("Số dòng đã lưu:", len([r for r in rows if any(c.strip() for c in r)]))
for i, r in enumerate(rows[:10]):
    if r and r[0]:
        print(f"{i+1}. {r[0]}: {(r[1] if len(r)>1 else '')[:120]}")

## Kết luận

Bạn vừa chạy một **bản demo FinSimAI hoạt động được** trên Colab + Google Sheets:

- ✅ **Math Engine**: GBM, tick size VN, trần–sàn, phí 0,15%, thuế bán 0,1%, **thanh toán T+2**.
- ✅ **AI Mentor Socratic**: Gemini + LLM-as-judge + 5 lớp bảo vệ + fallback deterministic.
- ✅ **Google Sheets làm database** qua tầng `SheetStore` trừu tượng.
- ✅ **Gamification** Điểm Kỷ luật (thưởng hành vi đúng, trừ khi cảm xúc/dồn vốn).

### Hướng tới production thật
Bản demo dùng Sheets vì tiện trên Colab. Trên nền tảng người dùng cuối:
1. Thay `SheetStore` bằng `PostgresStore` (cùng interface).
2. Chạy MarketSim/Mentor trên backend thật (FastAPI) với Postgres + Redis như hiện tại trong repo.
3. Chuyển kết nối Gemini về `integrations/gemini.py` và prompt về `prompts/mentor_prompts.yaml` của `apps/ai_engine`.
4. Bổ sung phòng chống look-ahead bias (đồng hồ mô phỏng + tem thời gian dữ liệu) như mô tả trong kế hoạch.

---
### Phụ lục: dùng Service Account (Cách B — cho nhiều người dùng/demo)
Thay ô "Kết nối Google Sheets" bằng code dưới đây (cần file JSON của service account trong Drive):

```python
# từ google.oauth2 import service_account
# from google.colab import drive; drive.mount('/content/drive')
# scope = ['https://spreadsheets.google.com/feeds','https://www.googleapis.com/auth/drive']
# creds = service_account.Credentials.from_service_account_file('/content/drive/MyDrive/xxx.json', scopes=scope)
# gc = gspread.authorize(creds); sh = gc.open_by_url(SHEET_URL)
```
Nhớ **share sheet cho email** trong file JSON của service account.